In [ ]:
# 🌌 HoroConsultant - Production Cloud Fine-Tuning Pipeline
import os
import sys
import subprocess

# Suppress PyDev / frozen modules debugger warnings
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
os.environ['PYTHONWARNINGS'] = 'ignore'

# 0. Set CUDA stability env vars FIRST (before any torch/bnb imports)
# Tesla T4 = sm_75 (Compute Capability 7.5) -- bfloat16 requires sm_80+
# BNB ops compiled for cu128 can lack sm_75 device code -> cudaErrorNoKernelImageForDevice
os.environ.setdefault('TORCH_CUDA_ARCH_LIST', '7.5')     # Target T4 arch
os.environ.setdefault('BNB_CUDA_VERSION', '121')          # Force CUDA 12.1 ABI-compatible BNB kernels
os.environ.setdefault('CUDA_MODULE_LOADING', 'LAZY')      # Lazy loading prevents JIT errors at import
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('\u2705 CUDA stability environment variables set (T4/sm_75 compatible)')

# 1. Load Secrets safely from Kaggle Secrets (individual try-except per key)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    for secret_key in ['HF_TOKEN', 'APP_SUPABASE_URL', 'APP_SUPABASE_KEY', 'GH_TOKEN']:
        try:
            val = user_secrets.get_secret(secret_key)
            if val:
                os.environ[secret_key] = val
                print(f'\u2705 Kaggle Secret loaded: {secret_key}')
        except Exception as e:
            print(f'\u2139\ufe0f Kaggle Secret note ({secret_key}): {e}')
except Exception as e:
    print(f'\u2139\ufe0f Kaggle Secrets Client not available: {e}')

# 2. Safe Git Clone / Pull with pure Python subprocess
target_dir = '/kaggle/working/HoroConsultant'
if not os.path.exists(target_dir):
    print('\ud83d\udce6 Cloning HoroConsultant repository...')
    subprocess.run(['git', 'clone', 'https://github.com/pphothidaen/HoroConsultant.git', target_dir], check=True)
else:
    print('\ud83d\udd04 Resetting and pulling latest updates...')
    subprocess.run(['git', '-C', target_dir, 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', target_dir, 'reset', '--hard', 'origin/main'], check=True)

os.chdir(target_dir)
if target_dir not in sys.path:
    sys.path.insert(0, target_dir)

# 3. Install Fine-Tuning Dependencies preserving Kaggle's pre-installed CUDA PyTorch
print('\ud83d\udce6 Checking pre-installed PyTorch & CUDA status...')
import torch
print(f'\u26a1 Kaggle PyTorch version: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    dev_name = torch.cuda.get_device_name(0)
    target_sm = f'sm_{cap[0]}{cap[1]}'
    print(f'   Device Count: {torch.cuda.device_count()}, Device 0: {dev_name} ({target_sm})')
    arch_list = torch.cuda.get_arch_list() if hasattr(torch.cuda, 'get_arch_list') else []
    if arch_list and not any(target_sm in a or f'{cap[0]}.{cap[1]}' in a for a in arch_list):
        print(f'\u26a0\ufe0f Pre-installed PyTorch wheel lacks binary for {target_sm} ({dev_name}). Installing cu121 PyTorch compatibility wheel...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--prefer-binary', 'torch==2.3.1', 'torchvision', 'torchaudio', '--index-url', 'https://download.pytorch.org/whl/cu121'], check=False)
    elif cap == (7, 5):
        print('\u2139\ufe0f Tesla T4 (sm_75) detected: float16 training, bitsandbytes 4-bit bypassed.')
print('\ud83d\udce6 Removing incompatible torchao & installing fine-tuning packages...')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--prefer-binary', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--prefer-binary', 'transformers>=4.40.0', 'peft>=0.10.0', 'bitsandbytes>=0.43.3', 'datasets>=2.18.0', 'trl>=0.12.0', 'huggingface_hub', 'accelerate'], check=True)
import torch
print(f'\u2705 Verified post-install PyTorch version: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')

# 4. Run Cloud Training Orchestrator with execution logging
# Pass the full environment (incl. CUDA stability vars) to subprocess
print('\ud83d\ude80 Launching Cloud Training Orchestrator...')
log_path = '/kaggle/working/train_execution.log'
train_env = os.environ.copy()
proc = subprocess.Popen([sys.executable, 'scripts/cloud_train_orchestrator.py', '--platform', 'KAGGLE_T4X2', '--base-model', 'Qwen/Qwen2.5-7B-Instruct', '--epochs', '3'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=train_env)
with open(log_path, 'w', encoding='utf-8') as log_f:
    for line in iter(proc.stdout.readline, ''):
        sys.stdout.write(line)
        log_f.write(line)
proc.wait()
if proc.returncode != 0:
    log_tail = ''
    if os.path.exists(log_path):
        try:
            with open(log_path, 'r', encoding='utf-8') as f:
                log_tail = ''.join(f.readlines()[-30:])
        except Exception:
            pass
    raise RuntimeError(f'\u274c Training orchestrator failed (exit code {proc.returncode}).\n--- Tail of train_execution.log ---\n{log_tail}')
print('\ud83c\udf89 Training pipeline completed successfully!')
